# Phase 09 — Quantization Config (QLoRA 4-bit NF4)
## CodeMentor-LLM
Configuring 4-bit quantization for memory efficient
fine-tuning of meta-llama/Meta-Llama-3-8B-Instruct.

## Goal
- Load Llama-3-8B in 4-bit NF4 quantization
- Measure memory footprint before and after
- Prepare model for k-bit training

# Libraries

In [ ]:
!pip install -q transformers==4.49.0 peft==0.14.0 bitsandbytes==0.45.3 accelerate==1.5.1

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in successfully")

# Check GPU memory before loading

In [ ]:
import torch

print("GPU Info:")
print(f"  GPU Name     : {torch.cuda.get_device_name(0)}")
print(f"  Total Memory : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"  Free Memory  : {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"  CUDA Version : {torch.version.cuda}")
print(f"  PyTorch      : {torch.__version__}")

# Define BitsAndBytes quantization config

In [ ]:
from transformers import BitsAndBytesConfig
import torch

# 4-bit NF4 quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("BitsAndBytesConfig defined successfully")
print(f"\nConfig details:")
print(f"  load_in_4bit          : {bnb_config.load_in_4bit}")
print(f"  bnb_4bit_quant_type   : {bnb_config.bnb_4bit_quant_type}")
print(f"  bnb_4bit_compute_dtype: {bnb_config.bnb_4bit_compute_dtype}")
print(f"  bnb_4bit_use_double_quant: {bnb_config.bnb_4bit_use_double_quant}")

#  Load model in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded successfully")

print("\nLoading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded successfully")
print(f"\nMemory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
'''
“Compress the model to 4-bit and load it into memory”
BUT the model is not ready for training yet
'''

#  Prepare model for k-bit training

In [ ]:
from peft import prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model) # prepares the model so it can be trained safely in 4-bit or 8-bit mode --> It makes a quantized model “trainable with LoRA

print("Model prepared for k-bit training successfully")
print(f"\nGradient checkpointing enabled: {model.is_gradient_checkpointing}")
print(f"Memory footprint after preparation: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
'''
Why we need prepare_model_for_kbit_training ?

saves memory during training
ensures only LoRA layers are trained
Casts some layers to FP32
ensures 4-bit weights don’t break training flow
'''

# Inspect model architecture

In [ ]:
# Print model architecture to identify attention layers
print("Model Architecture:")
print("=" * 10)
for name, module in model.named_modules():
    if "proj" in name and "layers.0" in name:
        print(f"  {name}: {module}")

# ============================================================================

# Phase 10 — LoRA Adapter Configuration
## CodeMentor-LLM
Applying LoRA adapter to quantized Llama-3-8B-Instruct
for parameter efficient fine-tuning.

## Goal
- Define LoRA config
- Apply adapter to model
- Print trainable parameter count
- Verify ~1% trainable parameters

# Define LoRA config

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# Define LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("LoRA Config defined successfully")
print(f"\nConfig details:")
print(f"  r (rank)          : {lora_config.r}")
print(f"  lora_alpha        : {lora_config.lora_alpha}")
print(f"  target_modules    : {lora_config.target_modules}")
print(f"  lora_dropout      : {lora_config.lora_dropout}")
print(f"  bias              : {lora_config.bias}")
print(f"  task_type         : {lora_config.task_type}")

# Apply LoRA adapter to model

In [ ]:
# Apply LoRA adapter to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# Save test checkpoint

In [ ]:
# Save test checkpoint to verify saving works
import os
os.makedirs("checkpoints/test", exist_ok=True)

model.save_pretrained("checkpoints/test")
print("Test checkpoint saved successfully")

# Verify files saved
files = os.listdir("checkpoints/test")
print(f"Saved files: {files}")